Ten skrypt oczyszcza i transformuje dane w hurtowni

In [0]:
%sql
MERGE INTO staging.car AS car
USING (
  SELECT
    Inventory.inventory_id,
    CASE WHEN Car.last_update > Inventory.last_update THEN Car.last_update ELSE Inventory.last_update END AS new_valid_to
  FROM landing.car AS Car
  INNER JOIN landing.inventory AS Inventory ON Car.car_id = Inventory.car_id
) AS src
ON car.car_inventory_id = src.inventory_id
AND (car.last_update < src.new_valid_to)
WHEN MATCHED THEN
  UPDATE SET
    car.last_update=src.new_valid_to,
    stag_update_date = CURRENT_TIMESTAMP()

In [0]:
%sql
INSERT INTO staging.car (car_inventory_id, car_id, car_producer, car_model, car_production_year, car_fuel_type, car_license_plates, car_current_rental_rate, car_purchase_price, car_sell_price, last_update,stag_insert_date, stag_update_date)
SELECT
    Inventory.inventory_id AS car_inventory_id,
    Car.car_id,
    Car.producer,
    Car.model,
    Inventory.production_year,
    Inventory.fuel_type,
    Inventory.license_plates,
    Car.rental_rate AS current_rental_rate,
    Inventory.purchase_price,
    Inventory.sell_price,
    CASE WHEN Car.last_update > Inventory.last_update THEN Car.last_update
        ELSE Inventory.last_update END AS last_update,
    CURRENT_TIMESTAMP() AS stag_insert_date,
    NULL AS dw_update_date
FROM landing.car Car
INNER JOIN landing.inventory Inventory ON Car.car_id = Inventory.car_id
LEFT JOIN dw.dim_car Dim_car ON Dim_car.car_inventory_id = Inventory.inventory_id AND Dim_car.is_active = true
WHERE Dim_car.car_inventory_id IS NULL;

In [0]:
%sql
UPDATE staging.car
SET car_fuel_type = 
  CASE LEFT(LOWER(car_fuel_type), 1)
    WHEN 'd' THEN 'Diesel'
    WHEN 'p' THEN 'Petrol'
    ELSE NULL END
WHERE car_fuel_type not in ('Diesel', 'Petrol');

In [0]:
%sql
UPDATE staging.car
SET car_purchase_price = 111000
WHERE car_purchase_price = 111;